In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from tabulate import tabulate

from main import (
    SEUIL_CROISSANCE_CA_MIN,
    SEUIL_DETTE_NETTE_EBE_MAX,
    SEUIL_GEARING_MAX,
    SEUIL_PE_MAX,
    SEUIL_ROE_MIN,
    SEUIL_VOLUME_MOYEN_MIN_VALEUR,
    expliquer_metriques,
    get_company_data,
)

In [2]:
expliquer_metriques()


--- Explication des Métriques et Seuils Utilisés ---

- Ticker:
  Symbole boursier de l'entreprise.

- Nom:
  Nom de l'entreprise.

- Secteur:
  Secteur d'activité de l'entreprise.

- Prix Actuel:
  Dernier prix de clôture de l'action.

- P/E (Price/Earnings Ratio):
  Ratio Cours/Bénéfices. Mesure la cherté d'une action. Un P/E bas (ici < 12.0) peut indiquer une sous-évaluation.

- ROE (Return On Equity):
  Rentabilité des Fonds Propres (Bénéfice Net / Capitaux Propres). Un ROE élevé (ici > 10%) est positif.

- P/B (Price/Book Ratio):
  Ratio Cours/Valeur Comptable.

- Croissance CA (%):
  Croissance du Chiffre d'Affaires sur la dernière année. Un chiffre positif (ici > 5%) indique une expansion.

- VE (M):
  Valeur d'Entreprise en millions.

- VE/CA:
  Ratio Valeur d'Entreprise / Chiffre d'Affaires. Bas est mieux.

- VE/EBE:
  Ratio Valeur d'Entreprise / Excédent Brut d'Exploitation (EBITDA). Bas est mieux.

- Dette Nette/EBE:
  Ratio Dette Nette / EBITDA. Mesure la capacité à rembou

In [3]:
from pathlib import Path

src_path = Path(r"F:\yfn\Euronext_Equities_2025-05-15.csv")
data_euronext = pd.read_csv(src_path, sep=";", skiprows=[1, 2, 3], dtype=str)
small_data_euronext = data_euronext.head(10)
symbols_2_check = data_euronext["Symbol"]

In [49]:
import yfinance as yf

# Choix de la source des tickers. Adaptez "example_list" si vous avez une autre méthode.
# Par exemple, si vous créez un CSV nommé "pme_tickers.csv" avec une colonne "Ticker":
# LISTE_TICKERS_PME = get_pme_tickers_from_source(source_type="csv_file")
# LISTE_TICKERS_PME = get_pme_tickers_from_source(df=data_euronext, source_type="csv_file")
LISTE_TICKERS_PME = ["CLA.PA", "FTE.F", "NBTX", "0X5.F", "DMS1.F", "A8D.F", "VCTP.XC", "HO.PA"]
if not LISTE_TICKERS_PME:
    print("Aucun ticker à analyser. Arrêt du script.")
    exit()

In [50]:
from tqdm.auto import tqdm

print(f"\nRécupération des données pour {len(LISTE_TICKERS_PME)} entreprises...")
all_data = []

# Use iteration on data extraction calling sevral times yf
# for i, ticker_sym in enumerate(LISTE_TICKERS_PME):
#     print(f"Chargement de {ticker_sym} ({i+1}/{len(LISTE_TICKERS_PME)})...")
#     data = get_company_data(ticker_sym)
#     all_data.append(data)
#     time.sleep(0.1) # Petite pause pour ne pas surcharger Yahoo Finance

# Use iteration on info directly from one call to yf
tickers_info = yf.Tickers(LISTE_TICKERS_PME)


Récupération des données pour 8 entreprises...


In [51]:
with tqdm(total=len(LISTE_TICKERS_PME), desc="Récupération des données") as pbar:
    for symbol, ticker_obj in tickers_info.tickers.items():
        print(symbol, ticker_obj.info)
        try:
            data = get_company_data(ticker_raw=ticker_obj)
            all_data.append(data)
        except Exception as e:
            print(f"{symbol}: Erreur lors de la récupération des infos ({e})")
        finally:
            pbar.update(1)

Récupération des données:   0%|          | 0/8 [00:00<?, ?it/s]

CLA.PA {'address1': '2 Rue Berthelot', 'address2': 'Immeuble Adamas', 'city': 'Courbevoie', 'zip': '92400', 'country': 'France', 'phone': '33 1 41 27 19 75', 'website': 'https://www.claranova.com', 'industry': 'Software - Application', 'industryKey': 'software-application', 'industryDisp': 'Software - Application', 'sector': 'Technology', 'sectorKey': 'technology', 'sectorDisp': 'Technology', 'longBusinessSummary': 'Claranova SE, a technology company, engages in personalized e-commerce, software publishing, and internet of things (IoT) management in France, the United States, the United Kingdom, Germany, other European countries, and internationally. It operates in three segments: PlanetArt, Avanquest, and myDevices. The PlanetArt segment offers photos, frames, personalized books, and FreePrints mobile applications. The Avanquest segment provides antivirus, ad blocker, cleaning, and optimization tools under the Adaware brand; document management tools under the SodaPDF brand; and photo

In [52]:
df_entreprises = pd.DataFrame(all_data)
# S'assurer que Ticker est bien l'index
if "Ticker" in df_entreprises.columns:
    df_entreprises.set_index("Ticker", inplace=True)

# Nettoyage initial : supprimer les lignes où le nom indique une erreur de chargement
df_entreprises = df_entreprises[~df_entreprises["Nom"].str.contains("Erreur", na=False)]

cols_to_round = [
    "Prix Actuel",
    "P/E",
    "ROE",
    "P/B",
    "Croissance CA (%)",
    "VE (M)",
    "VE/CA",
    "VE/EBE",
    "Dette Nette/EBE",
    "Gearing",
    "Vol. Moyen Val. (k)",
]
for col in cols_to_round:
    if col in df_entreprises.columns:
        df_entreprises[col] = pd.to_numeric(df_entreprises[col], errors="coerce").round(2)

print("\n--- Données Récupérées (Toutes les entreprises valides) ---")
# Afficher toutes les colonnes, même si larges pour la console
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 1000):
    print(tabulate(df_entreprises.reset_index(), headers="keys", tablefmt="pipe", showindex=False, floatfmt=".2f"))


# --- Screening des Entreprises Prometteuses (Filtre Strict) ---
# L'entreprise doit avoir les données pour les critères principaux et les respecter
df_prometteuses_strict = df_entreprises.dropna(
    subset=["P/E", "ROE", "Croissance CA (%)", "Dette Nette/EBE", "Gearing", "Vol. Moyen Val. (k)"]
)

df_prometteuses_strict = df_prometteuses_strict[
    (df_prometteuses_strict["P/E"] < SEUIL_PE_MAX)
    & (df_prometteuses_strict["P/E"] > 0)  # P/E doit être positif (entreprise rentable)
    & (df_prometteuses_strict["ROE"] > SEUIL_ROE_MIN)
    & (df_prometteuses_strict["Croissance CA (%)"] > (SEUIL_CROISSANCE_CA_MIN * 100))
    & (df_prometteuses_strict["Dette Nette/EBE"] < SEUIL_DETTE_NETTE_EBE_MAX)
    & (
        df_prometteuses_strict["Dette Nette/EBE"] > -5
    )  # Pour éviter des valeurs extrêmes négatives si EBE très petit et négatif
    & (df_prometteuses_strict["Gearing"] < SEUIL_GEARING_MAX)
    & (df_prometteuses_strict["Gearing"] > 0)  # Gearing doit être positif
    & (df_prometteuses_strict["Vol. Moyen Val. (k)"] > (SEUIL_VOLUME_MOYEN_MIN_VALEUR / 1000))
]

print(f"\n--- PME Potentiellement Prometteuses (Filtre Strict : {len(df_prometteuses_strict)} trouvées) ---")
if not df_prometteuses_strict.empty:
    cols_to_show_prometteuses = [
        "Nom",
        "Secteur",
        "P/E",
        "ROE",
        "Croissance CA (%)",
        "Dette Nette/EBE",
        "Gearing",
        "Vol. Moyen Val. (k)",
        "VE/CA",
        "VE/EBE",  # Garder pour info
    ]
    # S'assurer que les colonnes existent avant de les sélectionner
    cols_to_show_prometteuses = [col for col in cols_to_show_prometteuses if col in df_prometteuses_strict.columns]
    print(
        tabulate(
            df_prometteuses_strict[cols_to_show_prometteuses].reset_index(),
            headers="keys",
            tablefmt="pipe",
            showindex=False,
            floatfmt=".2f",
        )
    )
else:
    print("Aucune entreprise ne correspond à tous les critères stricts avec les données disponibles.")

# --- Visualisations avec Plotly ---
if not df_entreprises.empty:
    print("\nCréation des visualisations Plotly...")

    # Marquer les entreprises prometteuses pour la coloration
    df_entreprises["Est_Prometteuse"] = df_entreprises.index.isin(df_prometteuses_strict.index)

    # 1. Tableau interactif de toutes les données
    fig_table_all_cols = [
        "Nom",
        "Secteur",
        "Prix Actuel",
        "P/E",
        "ROE",
        "Croissance CA (%)",
        "Dette Nette/EBE",
        "Gearing",
        "Vol. Moyen Val. (k)",
        "VE/CA",
        "VE/EBE",
    ]
    # S'assurer que les colonnes existent
    fig_table_all_cols_exist = [col for col in fig_table_all_cols if col in df_entreprises.columns]

    # Préparer les données pour la table, s'assurer que l'index est une colonne pour l'affichage
    df_for_table = df_entreprises.reset_index()

    fig_table_all = go.Figure(
        data=[
            go.Table(
                header=dict(values=["Ticker"] + fig_table_all_cols_exist, fill_color="paleturquoise", align="left"),
                cells=dict(
                    values=[df_for_table["Ticker"]] + [df_for_table[col] for col in fig_table_all_cols_exist],
                    fill_color="lavender",
                    align="left",
                    format=[None] + [".2f"] * (len(fig_table_all_cols_exist) - 2),  # Format numerics
                ),
            )
        ]
    )
    fig_table_all.update_layout(
        title_text="Tableau de Bord des Entreprises Analysées", height=max(400, 50 + len(df_entreprises) * 35)
    )
    fig_table_all.show()

    # 2. Scatter Plot: P/E vs ROE
    df_plot_pe_roe = df_entreprises.dropna(subset=["P/E", "ROE"])
    if not df_plot_pe_roe.empty:
        fig_pe_roe = px.scatter(
            df_plot_pe_roe.reset_index(),
            x="P/E",
            y="ROE",
            text="Ticker",
            color="Est_Prometteuse",
            color_discrete_map={True: "green", False: "grey"},
            hover_data=["Nom", "Secteur", "Croissance CA (%)", "Dette Nette/EBE", "Vol. Moyen Val. (k)"],
            title="P/E vs. ROE des Entreprises",
        )
        fig_pe_roe.add_vline(
            x=SEUIL_PE_MAX, line_dash="dash", line_color="blue", annotation_text=f"P/E < {SEUIL_PE_MAX}"
        )
        fig_pe_roe.add_hline(
            y=SEUIL_ROE_MIN, line_dash="dash", line_color="blue", annotation_text=f"ROE > {SEUIL_ROE_MIN * 100:.0f}%"
        )
        fig_pe_roe.update_xaxes(
            range=[max(-5, df_plot_pe_roe["P/E"].min() - 5), min(50, df_plot_pe_roe["P/E"].max() + 5)]
        )  # Limiter plage P/E
        fig_pe_roe.update_traces(textposition="top center")
        fig_pe_roe.update_layout(height=700)
        fig_pe_roe.show()

    # 3. Bar Chart: Croissance du CA pour les entreprises prometteuses (si elles existent)
    if not df_prometteuses_strict.empty and "Croissance CA (%)" in df_prometteuses_strict.columns:
        df_plot_croissance = df_prometteuses_strict.dropna(subset=["Croissance CA (%)"]).sort_values(
            by="Croissance CA (%)", ascending=False
        )
        if not df_plot_croissance.empty:
            fig_croissance = px.bar(
                df_plot_croissance.reset_index(),
                x="Ticker",
                y="Croissance CA (%)",
                color="Croissance CA (%)",
                text="Croissance CA (%)",
                hover_data=["Nom", "P/E", "ROE"],
                title="Croissance du CA des Entreprises Prometteuses (Filtre Strict)",
            )
            fig_croissance.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
            fig_croissance.update_layout(
                uniformtext_minsize=8, uniformtext_mode="hide", height=max(400, 50 + len(df_plot_croissance) * 30)
            )
            fig_croissance.show()

    # 4. Scatter Plot: Dette Nette/EBE vs Gearing (pour les prometteuses)
    df_plot_debt = df_prometteuses_strict.dropna(subset=["Dette Nette/EBE", "Gearing"])
    if not df_plot_debt.empty:
        fig_debt = px.scatter(
            df_plot_debt.reset_index(),
            x="Dette Nette/EBE",
            y="Gearing",
            text="Ticker",
            color_discrete_sequence=["purple"],  # Toutes prometteuses ici
            hover_data=["Nom", "Secteur", "P/E", "ROE", "Vol. Moyen Val. (k)"],
            title="Endettement des Entreprises Prometteuses (Filtre Strict)",
        )
        fig_debt.add_vline(
            x=SEUIL_DETTE_NETTE_EBE_MAX,
            line_dash="dash",
            line_color="orange",
            annotation_text=f"Dette/EBE < {SEUIL_DETTE_NETTE_EBE_MAX}",
        )
        fig_debt.add_hline(
            y=SEUIL_GEARING_MAX, line_dash="dash", line_color="orange", annotation_text=f"Gearing < {SEUIL_GEARING_MAX}"
        )
        fig_debt.update_traces(textposition="top center")
        fig_debt.update_layout(height=700)
        fig_debt.show()

else:
    print("\nPas assez de données valides pour générer les graphiques Plotly.")

print("\nAnalyse terminée.")


--- Données Récupérées (Toutes les entreprises valides) ---
| Ticker   | Nom                             | Secteur                                  |   Prix Actuel |    P/E |    ROE |   P/B |   Croissance CA (%) |   VE (M) |   VE/CA |   VE/EBE |   Dette Nette/EBE |   Gearing |   Vol. Moyen Val. (k) |
|:---------|:--------------------------------|:-----------------------------------------|--------------:|-------:|-------:|------:|--------------------:|---------:|--------:|---------:|------------------:|----------:|----------------------:|
|          | Claranova SE                    | Software - Application                   |          2.50 | -12.78 | nan    | -8.46 |               -2.23 |   214.92 |    0.44 |     4.48 |              1.45 |     -9.83 |                362.59 |
|          | Orange S.A.                     | Telecom Services                         |         13.17 |  17.33 |   0.08 |  1.10 |                1.47 | 68482.78 |    1.70 |     5.29 |              2.35 |      1.


Analyse terminée.


In [16]:
fig_table_all.write_html("fig_table_all.html")
fig_croissance.write_html("fig_croissance.html")
fig_pe_roe.write_html("fig_peroe.html")
fig_debt.write_html("fig_debt.html")
fig_table_all_cols.write_html("fig_table_all_cols.html")

OSError: [Errno 22] Invalid argument: 'fig_table_all.html'